# return_values
**Prerequisites:** function_basics, parameters_and_arguments

**Outcome:** After this notebook you will know exactly what return does at the memory level, every form a return value can take, how to design return values that are predictable and safe to use, and the patterns that make callers write clean code.

## The Problem

In [ ]:
# what does this function actually return in each branch?
def process_job(job):
    if not job.get("id"):
        print("missing id")
    elif job["status"] == "done":
        return True
    elif job["status"] == "failed":
        return False

r1 = process_job({"id": "job_1", "status": "done"})
r2 = process_job({"id": "job_2", "status": "failed"})
r3 = process_job({"id": "",      "status": "pending"})
r4 = process_job({"id": "job_4", "status": "pending"})

print(r1, r2, r3, r4)
# r3 and r4 silently return None — the caller has no way to tell
# whether None means 'no id', 'unrecognised status', or 'not yet processed'

## The Concept

The `return` statement ends function execution and sends a value back to the caller. Python then destroys the function's local frame. The returned value is a reference to an existing object — Python never copies it. A function with no `return` statement, or a bare `return` with no value, returns `None`. A function can return any object: a single value, a tuple of multiple values, a dict, a list, another function, or `None`. The design of your return value is a contract with the caller — inconsistent return types across branches are one of the most common sources of bugs.

## Minimal Example

In [ ]:
def add(a, b):
    return a + b

result = add(10, 20)
print(result)        # 30
print(type(result))  # <class 'int'>

## How Python Does It

When Python executes `return value` it evaluates the expression, stores the resulting object reference in the frame's return slot, destroys the local namespace, pops the stack frame, and hands the reference back to the caller's frame. The caller can then bind it to a name, pass it to another function, or discard it. If no `return` is executed — because the function body finishes naturally or hits a bare `return` — Python places `None` in the return slot automatically.

In [ ]:
# proving that return passes a reference, not a copy
def get_jobs():
    jobs = ["job_1", "job_2", "job_3"]
    return jobs

result = get_jobs()
result.append("job_4")
print(result)   # ["job_1", "job_2", "job_3", "job_4"]
# the returned list is the same object that lived inside the function

## Building Up

In [ ]:
# no return statement — implicitly returns None
def log_event(event):
    print(f"[LOG] {event}")

result = log_event("boot")
print(result)           # None
print(result is None)   # True

In [ ]:
# bare return — exits early, also returns None
def validate_and_log(record):
    if not record.get("id"):
        print("skipping record with no id")
        return                        # exits here, returns None
    print(f"processing {record['id']}")

validate_and_log({"id": "",      "status": "pending"})
validate_and_log({"id": "job_1", "status": "pending"})

In [ ]:
# returning multiple values — Python packs them into a tuple
def get_stats(values):
    return min(values), max(values), sum(values) / len(values)

low, high, avg = get_stats([90, 120, 150, 80, 200])
print(f"min={low}  max={high}  avg={avg}")

# without unpacking — it is a tuple
result = get_stats([90, 120, 150, 80, 200])
print(type(result))   # <class 'tuple'>
print(result)         # (80, 200, 128.0)

In [ ]:
# returning a dict — named fields, self-documenting, extensible
def process_record(record):
    return {
        "id":       record["id"],
        "status":   "processed",
        "original": record["status"]
    }

result = process_record({"id": "job_1", "status": "pending"})
print(result["status"])    # access by name, not by position

In [ ]:
# returning None explicitly to signal absence
def find_job(jobs, job_id):
    for job in jobs:
        if job["id"] == job_id:
            return job
    return None            # explicit — caller knows to check for None

jobs = [{"id": "job_1"}, {"id": "job_2"}]
found = find_job(jobs, "job_3")

if found is None:
    print("job not found")
else:
    print(found)

In [ ]:
# returning a function — functions are objects
def make_multiplier(factor):
    def multiply(value):
        return value * factor
    return multiply        # return the function object, not its result

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10))   # 20
print(triple(10))   # 30

In [ ]:
# chaining return values
def load(source):
    return [{"id": f"job_{i}", "value": i} for i in range(len(source))]

def filter_positive(records):
    return [r for r in records if r["value"] > 0]

def total_value(records):
    return sum(r["value"] for r in records)

result = total_value(filter_positive(load(["a", "b", "c", "d"])))
print(result)   # 1 + 2 + 3 = 6

## Where It Breaks

In [ ]:
# inconsistent return type — sometimes a list, sometimes None
def get_failed_jobs(jobs):
    failed = [j for j in jobs if j["status"] == "failed"]
    if failed:
        return failed   # returns a list
    # implicit None when no failures — caller must handle two different types

jobs = [{"id": "job_1", "status": "done"}]
result = get_failed_jobs(jobs)

try:
    print(len(result))   # TypeError: object of type 'NoneType' has no len()
except TypeError as e:
    print(e)

In [ ]:
# return inside a finally block suppresses the original return
def risky():
    try:
        return "from try"
    finally:
        return "from finally"   # this overwrites the try return

print(risky())   # "from finally" — the try return is discarded silently

## The Fix

In [ ]:
# fix: always return the same type from every branch
def get_failed_jobs(jobs):
    return [j for j in jobs if j["status"] == "failed"]  # always a list
    # caller can check len(result) == 0 instead of result is None

jobs = [{"id": "job_1", "status": "done"}]
result = get_failed_jobs(jobs)
print(len(result))   # 0 — safe, predictable

In [ ]:
# fix: use a result dict to return structured status from multi-branch functions
def process_job(job):
    if not job.get("id"):
        return {"ok": False, "reason": "missing id"}
    if job["status"] == "done":
        return {"ok": True,  "reason": None}
    if job["status"] == "failed":
        return {"ok": False, "reason": "job failed"}
    return {"ok": False, "reason": f"unknown status: {job['status']}"}

for job in [
    {"id": "job_1", "status": "done"},
    {"id": "job_2", "status": "failed"},
    {"id": "",      "status": "pending"},
    {"id": "job_4", "status": "pending"},
]:
    r = process_job(job)
    print(f"{job.get('id') or 'no-id':8} ok={r['ok']}  reason={r['reason']}")

## In a Real System

In [ ]:
# a pipeline stage that always returns a consistent result envelope
# callers can always access .ok, .data, and .error without type checking
def make_result(ok, data=None, error=None):
    return {"ok": ok, "data": data, "error": error}

def extract(source_path):
    if not source_path:
        return make_result(ok=False, error="source_path is empty")
    records = [{"id": f"job_{i}"} for i in range(3)]
    return make_result(ok=True, data=records)

def transform(result):
    if not result["ok"]:
        return result                        # pass failure downstream unchanged
    enriched = [{**r, "status": "transformed"} for r in result["data"]]
    return make_result(ok=True, data=enriched)

r1 = transform(extract(""))
r2 = transform(extract("/data/jobs.csv"))

print(r1)
print(r2)

## Performance

Return passes a reference — O(1) regardless of object size. Returning a tuple of multiple values creates one small tuple allocation — negligible in almost all cases. Returning a large data structure does not copy it. The cost is always in building the return value, not in the return itself. If you find yourself returning very large lists from many nested calls, consider generators instead — they yield one item at a time without building the full collection in memory.

## Anti-Pattern

In [ ]:
# anti-pattern: returning different types from different branches
def get_job(jobs, job_id):
    for job in jobs:
        if job["id"] == job_id:
            return job            # returns a dict
    return "not found"            # returns a string — different type

result = get_job([], "job_1")
# caller cannot write result["status"] — sometimes dict, sometimes str

# correct: return None for absence, same type for presence
def get_job(jobs, job_id):
    for job in jobs:
        if job["id"] == job_id:
            return job       # dict
    return None              # absence is None, not a string

result = get_job([], "job_1")
if result is None:
    print("not found")
else:
    print(result["status"])

## Interview Signals

- What does Python return if a function has no `return` statement?
- When you return multiple values from a function what type does Python use to pack them?
- Why is returning different types from different branches a design problem?
- What happens when `return` is used inside a `finally` block?

## Exercise

In [ ]:
def partition_jobs(jobs):
    """
    Takes a list of job dicts, each with a 'status' key.
    Returns three values in this order:
        done    — list of jobs where status == 'done'
        failed  — list of jobs where status == 'failed'
        pending — list of jobs where status is anything else

    Each return value must always be a list, never None.
    Bug: the function body is missing — implement it.
    """
    pass


jobs = [
    {"id": "job_1", "status": "done"},
    {"id": "job_2", "status": "failed"},
    {"id": "job_3", "status": "pending"},
    {"id": "job_4", "status": "done"},
    {"id": "job_5", "status": "running"},
]

done, failed, pending = partition_jobs(jobs)

assert len(done)    == 2,                          f"got {len(done)}"
assert len(failed)  == 1,                          f"got {len(failed)}"
assert len(pending) == 2,                          f"got {len(pending)}"
assert done[0]["id"]    == "job_1"
assert failed[0]["id"]  == "job_2"
assert pending[0]["id"] == "job_3"

empty_done, empty_failed, empty_pending = partition_jobs([])
assert empty_done    == []
assert empty_failed  == []
assert empty_pending == []

print("all assertions passed")